In [1]:
import os
import torch
from transformers import AutoTokenizer
import yaml
import sys
sys.path.append("/home/wengang/vanilla/routing_decision")
sys.path.append("/home/wengang/vanilla/routing_decision/customized_models")
sys.path.append("/home/wengang/vanilla/routing_decision/tools")
from analyze import *
###-------- Basic settings --------####
with open("/home/wengang/vanilla/routing_decision/config.yaml", "r") as f:
    data = yaml.safe_load(f)
output_dir = data["result_path"] + "test_olmoe/test/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
torch.set_default_device("cpu") # or "cpu"
torch.set_grad_enabled(False)
####-------- Model settings --------####
model_id = "gpt2" # "allenai/OLMoE-1B-7B-0125"
n_layers = 12
n_dim = 768
n_heads = 12
n_head_dim = 64
####-------- Model loading --------####
from customized_models.modeling_gpt2_customized import GPT2LMHeadModel
model = GPT2LMHeadModel.from_pretrained(model_id, attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(model_id)
####-------- Prompt settings --------####
prompt_orig = "After the lunch, Matthew and Andrew went to the hospital. Matthew gave a basketball to Andrew" # Matthew: 9308, Andrew: 6858, S1[4]; IO[6]; S2[12]; end[16]
prompt_new = "After the lunch, Erin and Tiffany went to the hospital. Jesse gave a basketball to Tiffany" # Erin: 28894, Tiffany: 40928, Jesse: 18033
# ####-------- Experiments --------####
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model.resize_token_embeddings(len(tokenizer))
# batch_token = tokenizer(prompt_orig, return_tensors="pt", padding=True) # for test
# batch_token = tokenizer(prompt_new, return_tensors="pt", padding=True) # for test
prompt_dict_ls_ORIG = [{"text":prompt_orig, "IO_token_id":[6858], "S_token_id":[9308], "S2_token_id":[9308], "END_token_pos":16}] * 10
prompt_dict_ls_NEW = [{"text":prompt_new, "IO_token_id":[40928], "S_token_id":[28894], "S2_token_id":[18033], "END_token_pos":16}] * 10


/home/wengang/.conda/envs/paper_routing_decision/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚨 `patching` is part of GPT2Model.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/wengang/vanilla/routing_decision/customized_models/modeling_gpt2_customized.py.
🚨 `patching` is part of GPT2LMHeadModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /home/wengang/vanilla/routing_decision/customized_models/modeling_gpt2_customized.py.


The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [2]:
# find name mover and negative name mover heads at Token END
send_info = {"token_pos_ls":([16]*10)} # example: [ i[END_token_pos] for i in prompt_dict_ls_ORIG ]
recv_info = {"type":"l", "token_pos_ls":([16]*10)}
path_patching(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, send_info, recv_info, output_dir, n_layers, n_heads, bsz=20, demo_now=True)

100%|██████████| 1/1 [00:26<00:00, 26.02s/it]


In [3]:
# find s2-inhibition heads at Token S2
send_info = {"token_pos_ls":([12]*10)}
recv_info = {"type":"qkv", "token_pos_ls":([12]*10), "head_pos":[(7, 3), (7, 9), (8, 6), (8, 10)]}
path_patching(prompt_dict_ls_ORIG, prompt_dict_ls_NEW, model, tokenizer, send_info, recv_info, output_dir, n_layers, n_heads, bsz=20, demo_now=True)

100%|██████████| 1/1 [01:47<00:00, 107.20s/it]
